# Dataset generation for Electricity Price Forecasting

This notebook generates the dataset for electricity price forecasting by:
1. Fetching data from ENTSOE API (2023-02-01 to 2026-02-01)
2. Including original exogenous variables (from paper Jesus Lago: FR Generation Forecast, FR Load Forecast)
3. Adding additional potentially useful variables
4. Validating variables using correlation and mutual information

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))

from data.dataset_generator import ENTSOEDatasetGenerator
import pandas as pd

# Initialize generator
generator = ENTSOEDatasetGenerator()

# fetch the dataset (2023-02-01 to 2026-02-01)
df = generator.fetch_full_dataset(
    start_date="2023-02-01",
    end_date="2026-02-01",
    verbose=True
)

## Add time-based and derived features


In [ ]:
# Add time-based features
print("Adding time-based features...")
df = generator.add_time_features(df)

# Add derived features
print("Adding derived features...")
df = generator.add_derived_features(df)

print(f"\nDataset shape after feature engineering: {df.shape}")
print(f"Columns: {list(df.columns)}")

## Validate exogenous variables

In [ ]:
# Validate variables using correlation and mutual information
validation_results = generator.validate_variables(
    df,
    train_split_date="2023-02-01",
    top_n=20
)

# Display top variables
print("\nTop variables by correlation and mutual information:")
print(validation_results.head(15))

## Select variables and save final dataset

In [ ]:
# Select the best variables (exogenous variables with high correlation/MI) (excluding time features which are always picked)
time_features = [
    'Hour', 'DayOfWeek', 'Month', 'IsWeekend', 'DayOfYear', 'WeekOfYear',
    'Hour_sin', 'Hour_cos', 'DayOfWeek_sin', 'DayOfWeek_cos',
    'Month_sin', 'Month_cos'
]

# Get best exogenous variables
top_exog = validation_results[
    ~validation_results['Variable'].isin(time_features + ['Prices'])
].head(15)['Variable'].tolist()

# Combine Prices + time features + top exogenous variables
selected_vars = ['Prices'] + time_features + top_exog

print(f"\nSelected {len(selected_vars)} variables:")
print(f"  - Target: Prices")
print(f"  - Time features: {len(time_features)}")
print(f"  - Exogenous variables: {len(top_exog)}")
print(f"\nExogenous variables:")
for var in top_exog:
    print(f"  - {var}")

# Save datasets
from pathlib import Path

output_dir = Path.cwd().parent / "data" / "datasets"
output_dir.mkdir(parents=True, exist_ok=True)

# Save full dataset
generator.save_dataset(
    df,
    output_dir / "BE_ENTSOE_FULL.csv",
    selected_variables=None  # Save all variables
)

# Save selected variables dataset
generator.save_dataset(
    df,
    output_dir / "BE_ENTSOE.csv",
    selected_variables=selected_vars
)

# Save validation results
validation_results.to_csv(
    output_dir / "variable_validation.csv",
    index=False
)

print("\n" + "="*80)
print("Dataset generation complete!")
print("="*80)
print(f"\nFiles created:")
print(f"  1. {output_dir / 'BE_ENTSOE_FULL.csv'} - All variables")
print(f"  2. {output_dir / 'BE_ENTSOE.csv'} - Selected variables (for refactored code)")
print(f"  3. {output_dir / 'variable_validation.csv'} - Validation metrics")